# H1 Encoding

Frozen before the run.

**Claim.** Encoding paragraphs separately costs accuracy only when the chain crosses workers, and the cost grows with crossings, even though the orchestrator holds every paragraph's KV.

**Arms** (thinking off):
- `b1`: all paragraphs as text in one context.
- `b1_mask`: B1 with a block-diagonal attention mask and true positions.
- `k0`: paragraph KV, each worker encoded right after S, keys shifted into place.

The decomposition:
- B1 − B1-mask = cost of missing cross-paragraph attention.
- B1-mask − K0 = cost of encoding each block at local positions.

**Gates (in `tests/test_gates.py`, must pass first):**
- K0 with one worker reproduces B1.
- B1-mask equals an explicit 4D block mask.

**The 0-crossing control is BB-type.** AA/AAA/AAAA put all gold in the first stitched block. A causal first block never sees later blocks, even in B1, so K0 encodes the gold exactly like B1. Only the distractor block differs, so "B1 − K0 ≈ 0" there is close to guaranteed. It's kept only as a reference. BB/BBB/BBBB put all gold in the second block, which is encoded separately and RoPE-shifted but has nothing to cross. That's the real control: it separates "crossing workers" from "sitting in a separately encoded block".

**Prediction.**
- B1 − K0 ≈ 0 under BB-type placements (TOST within ±2 EM).
- The gap grows with crossings: gap(c) − gap(BB) > 0 for every c ≥ 1, on the same questions.
- Most of the gap is B1 − B1-mask (missing attention), not B1-mask − K0.

**Decision rule.** H1 holds if the BB TOST passes and the paired interaction CI (vs BB) excludes 0 at the highest crossing level, on both toy_d2 and MuSiQue. If the gap is already present under BB, the cost is separate encoding itself, not composition across workers, and the claim has to change.

**Pilot (Qwen2.5, EM):**

| Setting | B1 | K0 |
|---|---|---|
| toy 3B, no reasoning, 0 crossings | 100 | 96 |
| toy 3B, no reasoning, 1 crossing | 90 | 36 |
| toy 7B, forced reasoning, 1 crossing | 99 | 85 |
| toy 7B, forced reasoning, 2 crossings | 97 | 32 |

MuSiQue 7B, F1: 2-step .414 vs .294; 3-step .415 vs .313. `toy_d0` is the pilot's toy setting (2 shards, no distractors); `toy_d2` is the new one (2 distractor shards, so AA and BB are possible).

**Not built yet:** CacheBlend-style partial recompute (bridge tokens vs random tokens).

In [ ]:
# Settings. Headless full run: see README "Running".
MODEL = "Qwen/Qwen3-4B"
DTYPE = "bfloat16"
SMOKE = True        # True: 2 questions per cell, to read raw outputs first. False: the full design.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from latent_dispatch import analysis, arms, data
from latent_dispatch.model import Model
from latent_dispatch.results import load_results, results_path, run
from latent_dispatch.score import contains_answer, normalize

## Questions and arms

In [ ]:
qs = data.load("toy") + data.load("toy_d0") + data.load("musique")
if SMOKE:
    qs = data.per_cell(qs, 2)
print(len(qs), "questions")

ARMS = {
    "b1": (arms.b1, {}),
    "b1_mask": (arms.b1_mask, {}),
    "k0": (arms.k0, {}),
}
BB = ["BB", "BBB", "BBBB"]       # the 0-crossing control
AA = ["AA", "AAA", "AAAA"]       # reference only: gold KV identical to B1 by construction

In [ ]:
model = Model.load(MODEL, dtype=DTYPE)
for name, (arm, options) in ARMS.items():
    run(model, qs, name, arm, **options)

In [ ]:
df = load_results(qs, ARMS, MODEL, DTYPE)
if SMOKE:
    analysis.peek(df)

## Accuracy

In [ ]:
analysis.accuracy(df, by=("dataset", "qtype", "placement"))

## Paired gaps: B1 − K0, and its two parts

In [ ]:
for a, b in (("b1", "k0"), ("b1", "b1_mask"), ("b1_mask", "k0")):
    print(f"{a} − {b}")
    display(analysis.paired(df, a, b, by=("dataset", "crossings", "placement")))

## Is B1 − K0 ≈ 0 at the BB control? (TOST, ±2 EM)

In [ ]:
display(analysis.equivalent(df, "b1", "k0", placement=BB))
print("reference, AA-type (gold KV identical, only the distractor block differs):")
display(analysis.equivalent(df, "b1", "k0", placement=AA))

## Does the gap grow with crossings? gap(c) − gap(BB), same questions

In [ ]:
for ds in ("toy_d2", "musique"):
    print(ds)
    display(analysis.crossing_trend(df, "b1", "k0", ds, base_placements=BB))

In [ ]:
t = analysis.paired(df[~df.placement.isin(AA) & (df.qtype != "hop0")], "b1", "k0",
                   by=("dataset", "crossings")).reset_index()
fig, ax = plt.subplots(figsize=(5, 3.5))
for ds, g in t.groupby("dataset"):
    ax.errorbar(g.crossings, g["diff"], yerr=[g["diff"] - g.lo, g.hi - g["diff"]], marker="o", capsize=3, label=ds)
ax.axhline(0, color="gray", lw=0.8)
ax.set(xlabel="crossings (0 = BB-type)", ylabel="B1 − K0 (EM points, 95% CI)", xticks=sorted(t.crossings.unique()))
ax.legend()
plt.show()

## Findings

_(fill in after the full run)_